# Taxi M/L: rerun with solvable conditional-door layouts

Each seed retains its identity and exactly `round(rho * number_of_doors)` conditional doors.
Invalid draws are rejected using the same seeded RNG until every start can finish within
200 actions. Rejection sampling conditions the map distribution on solvability; it does
not guarantee that any learned agent succeeds. All methods use the same accepted layout.

M: 30 seeds, all 1,200 starts, three datasets, 50/150 trajectories, rho 0/0.25/0.5.
L: 10 seeds, 1,000 sampled starts, drivers/random_mix, 150/400 trajectories, rho 0/0.5.
All nine methods are included. Historical CSVs are retained; the reruns use new filenames.


In [1]:
from pathlib import Path
import os
import subprocess
import sys
import time
import pandas as pd

PROJECT = next(p for base in [Path.cwd(), *Path.cwd().parents]
               for p in (base, base / "research_project")
               if (p / "src" / "envs.py").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))


## Regression tests

The new tests check all 30 M and 10 L seeds against shortest paths in the actual MDP,
reproducibility, preservation of valid first draws, fixed door counts, and bounded failure.


In [2]:
check = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                       cwd=PROJECT, capture_output=True, text=True)
print(check.stdout)
if check.stderr:
    print(check.stderr)
check.check_returncode()


..........................                                               [100%]
26 passed in 5.61s



## Run experiments

Enable `RUN_EXPERIMENTS` only for a new run and choose unused output filenames.
The two sizes run concurrently with four M workers and two L workers.
Progress is written to separate log files. A failed run raises an error; no seeds
or methods are removed to make the comparison complete.


In [3]:
RUN_EXPERIMENTS = False
OUTPUTS = {size: PROJECT / "results" / f"main_{size}_solvable.csv" for size in ("M", "L")}

if RUN_EXPERIMENTS:
    for output in OUTPUTS.values():
        if output.exists():
            raise FileExistsError(f"Choose an unused output filename: {output}")
    options = {
        "M": ["--seeds", "30", "--rhos", "0", "0.25", "0.5", "--datasets",
              "drivers", "noisy_expert", "random_mix", "--n-traj", "50", "150", "--jobs", "4"],
        "L": ["--seeds", "10", "--rhos", "0", "0.5", "--datasets",
              "drivers", "random_mix", "--n-traj", "150", "400", "--eval-starts", "1000", "--jobs", "2"],
    }
    processes, logs = {}, {}
    try:
        for size, extra in options.items():
            logs[size] = OUTPUTS[size].with_suffix(".log").open("w")
            processes[size] = subprocess.Popen(
                [sys.executable, "-m", "experiments.run_main", "--size", size,
                 *extra, "--out", str(OUTPUTS[size])],
                cwd=PROJECT, stdout=logs[size], stderr=subprocess.STDOUT,
                env={**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"},
            )
        pending = set(processes)
        while pending:
            for size in list(pending):
                status = processes[size].poll()
                if status is not None:
                    pending.remove(size)
                    if status:
                        raise RuntimeError(f"Taxi {size} failed; inspect its .log file")
                    print(f"Taxi {size} completed", flush=True)
            if pending:
                time.sleep(5)
    finally:
        for process in processes.values():
            if process.poll() is None:
                process.terminate()
        for stream in logs.values():
            stream.close()


## Verify complete comparisons

Every method must have the exact expected seed set in every dataset/rho/budget cell.
The generator certificate must be identical across methods and datasets for a given
size/rho/seed, and must certify completion within the episode horizon.


In [4]:
from experiments.common import ALL_AGENTS
from src.envs import MAX_STEPS

frames = {}
for size, expected_seeds, expected_rows in [("M", 30, 4860), ("L", 10, 720)]:
    frame = pd.read_csv(OUTPUTS[size])
    keys = ["env", "rho", "dataset", "n_traj", "seed", "agent"]
    assert len(frame) == expected_rows
    assert not frame.duplicated(keys).any()
    assert set(frame.agent) == set(ALL_AGENTS)
    assert frame.require_solvable.all()
    assert frame.max_optimal_steps.between(1, MAX_STEPS).all()
    assert frame.n_eval_starts.eq(1200 if size == "M" else 1000).all()
    for _, group in frame.groupby(["rho", "dataset", "n_traj", "agent"]):
        assert set(group.seed) == set(range(expected_seeds))
    for _, group in frame.groupby(["rho", "seed"]):
        assert group.layout_attempts.nunique() == 1
        assert group.max_optimal_steps.nunique() == 1
    frames[size] = frame
    print(f"Taxi {size}: {len(frame)} rows; all 9 methods have all {expected_seeds} seeds in every condition")
    layout = frame[frame.rho == 0.5].drop_duplicates("seed")
    display(layout[["seed", "layout_attempts", "max_optimal_steps"]].reset_index(drop=True))


Taxi M: 4860 rows; all 9 methods have all 30 seeds in every condition


,seed,layout_attempts,max_optimal_steps
0,0,1,47
1,1,1,41
2,2,1,45
3,3,1,45
4,4,1,43
5,5,1,38
6,6,2,43
7,7,1,47
8,8,1,47
9,9,1,41


Taxi L: 720 rows; all 9 methods have all 10 seeds in every condition


,seed,layout_attempts,max_optimal_steps
0,0,1,99
1,1,1,90
2,2,1,78
3,3,2,87
4,4,4,78
5,5,1,81
6,6,1,89
7,7,3,99
8,8,1,84
9,9,1,91


## Main comparison at rho = 0.5

Report all seeds without feasibility filtering. Positive return difference favors Ours;
positive query difference means Ours asks for help in more episodes (percentage points).
These are descriptive means; paired intervals are regenerated by the existing analysis script.


In [5]:
for size, frame in frames.items():
    subset = frame[(frame.rho == 0.5) & frame.agent.isin(["Ours", "RECO-greedy"])]
    print(f"Taxi {size}")
    display(subset.groupby(["dataset", "n_traj", "agent"])[
        ["ret", "queries_per_ep", "pct_ep_with_query", "pct_delivered"]].mean().round(3))


Taxi M


ret  queries_per_ep  pct_ep_with_query  \
dataset      n_traj agent                                                     
drivers      50     Ours          -8.796           0.240              6.853   
                    RECO-greedy   -7.349           0.225              6.406   
             150    Ours          -7.241           0.040              2.217   
                    RECO-greedy   -6.019           0.036              1.892   
noisy_expert 50     Ours          -7.387           0.166              3.450   
                    RECO-greedy   -6.118           0.166              3.450   
             150    Ours          -6.560           0.001              0.100   
                    RECO-greedy   -5.331           0.001              0.100   
random_mix   50     Ours        -102.169           0.252              2.222   
                    RECO-greedy -347.421           0.252              2.222   
             150    Ours          -7.005           0.000              0.000   
                    RECO-greedy  -38.456           0.000              0.000   

                                 pct_delivered  
dataset      n_traj agent                       
drivers      50     Ours               100.000  
                    RECO-greedy        100.000  
             150    Ours               100.000  
                    RECO-greedy        100.000  
noisy_expert 50     Ours               100.000  
                    RECO-greedy        100.000  
             150    Ours               100.000  
                    RECO-greedy        100.000  
random_mix   50     Ours                87.500  
                    RECO-greedy         74.958  
             150    Ours                99.722  
                    RECO-greedy         97.436

Taxi L


ret  queries_per_ep  pct_ep_with_query  \
dataset    n_traj agent                                                     
drivers    150    Ours         -33.492           1.646              17.25   
                  RECO-greedy  -28.795           1.543              13.18   
           400    Ours         -33.060           0.075               2.15   
                  RECO-greedy  -26.412           0.076               2.25   
random_mix 150    Ours        -209.805           1.548               6.81   
                  RECO-greedy -602.531           1.548               6.81   
           400    Ours         -56.382           0.023               0.12   
                  RECO-greedy -416.986           0.023               0.12   

                               pct_delivered  
dataset    n_traj agent                       
drivers    150    Ours                100.00  
                  RECO-greedy         100.00  
           400    Ours                100.00  
                  RECO-greedy         100.00  
random_mix 150    Ours                 75.26  
                  RECO-greedy          57.15  
           400    Ours                 95.78  
                  RECO-greedy          70.88